In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [14]:
combined_data = pd.read_csv('Combined_dataset.csv')

print(combined_data.dtypes)


/var/folders/qn/t0tfgsq57qn5ld54tyxxhyjw0000gn/T/ipykernel_86545/1237917507.py:1: DtypeWarning: Columns (2,6,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_data = pd.read_csv('Combined_dataset.csv')


track_id             object
artists              object
album_name           object
track_name           object
popularity          float64
duration_ms           int64
explicit             object
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature      float64
year                float64
genre                object
release_date         object
dtype: object


In [15]:
feature_cols = [
    'energy',
    'danceability',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature'
]

features_table = combined_data[feature_cols]



In [16]:
scaler = StandardScaler()

scaled_values =  scaler.fit_transform(features_table)
features_table = pd.DataFrame(
    scaled_values,
    columns=feature_cols,
    index=features_table.index
)

features_table.head()


,energy,danceability,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,-0.384564,0.847400,-1.195766,0.564378,-1.372290,0.448245,-0.945123,-0.717243,0.764716,1.005964,-1.036677,0.272317
1,-1.402040,-0.511468,-1.195766,-1.042546,0.728709,-0.100353,1.430091,-0.717230,-0.581023,-0.652280,-1.379373,0.272317
2,-0.736369,-0.415923,-1.477778,0.106614,0.728709,-0.269785,-0.471572,-0.717246,-0.497242,-1.196392,-1.417395,0.272317
3,-1.769021,-1.328912,-1.477778,-1.238643,0.728709,-0.429347,1.379486,-0.717055,-0.418697,-1.111258,2.046635,-1.658343
4,-0.446647,0.539531,-0.913755,0.114733,0.728709,-0.295282,0.218247,-0.717246,-0.675801,-1.022424,0.015993,0.272317


In [ ]:
# Create cleaned columns for matching
def clean_name(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

combined_data["artist_match"] = clean_name(combined_data["artists"])
combined_data["song_match"] = clean_name(combined_data["track_name"])

listening_data["artist_match"] = clean_name(listening_data["artist"])
listening_data["song_match"] = clean_name(listening_data["song"])

# Choose features for the listener profiles
profile_features = [
    "energy",
    "danceability",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

# Keep one track per artist/song pair to avoid multiplying listening records
track_features = (
    combined_data
    .dropna(subset=["artist_match", "song_match"])
    .drop_duplicates(subset=["artist_match", "song_match"])
    [["artist_match", "song_match"] + profile_features]
)

listening_with_features = listening_data.merge(
    track_features,
    on=["artist_match", "song_match"],
    how="left",
    indicator=True
)

print(listening_with_features["_merge"].value_counts())
print("Match rate:", listening_with_features["_merge"].eq("both").mean())



_merge
left_only     2759319
both          2015397
right_only          0
Name: count, dtype: int64
Match rate: 0.42209777502997037


In [22]:

matched_listening =  listening_with_features[  
    listening_with_features['_merge'] == 'both'

]

listener_profiles = ( 
    matched_listening 
    .groupby('userid')[profile_features]
    .mean()
)

listener_profiles['matched_listens'] = ( 
    matched_listening.groupby('userid').size()
)

display(listener_profiles.head())


,energy,danceability,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,matched_listens
userid,,,,,,,,,,
user_000001,0.593626,0.583878,-10.424055,0.077572,0.295842,0.510802,0.188072,0.442833,119.390172,1245
user_000002,0.658353,0.481868,-7.283201,0.055336,0.259227,0.073806,0.183350,0.473366,126.316502,7469
user_000003,0.738791,0.527970,-7.010156,0.074176,0.130110,0.146446,0.202033,0.498061,126.647947,2603
user_000004,0.586232,0.524946,-8.797377,0.049205,0.314764,0.169038,0.177655,0.440399,122.172095,1830
user_000005,0.679131,0.608772,-7.920507,0.055657,0.151537,0.353386,0.173399,0.466178,119.441358,1748


In [ ]:
from sklearn.preprocessing import StandardScaler 
from sklearn.cluster import KMeans 
from sklearn.metrics import silhouette_score


x = listener_profiles[profile_features]

scaler = StandardScaler()
x_scaled =  scaler.fit_transform(x)




sample_size =  min(20_000, len(x_scaled))

sample_index  = listener_profiles.sample(
    n=sample_size, 
    random_state = 42
).index 

X_sample = x_scaled[
    listener_profiles.index.get_indexer(sample_index)
]

results = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_sample)

    score = silhouette_score(X_sample, labels)

    results.append({
        "clusters": k,
        "silhouette_score": score,
        "inertia": kmeans.inertia_
    })

results_df = pd.DataFrame(results)
display(results_df)



final_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

listener_profiles["cluster"] = final_kmeans.fit_predict(x_scaled)

cluster_summary = (
    listener_profiles
    .groupby("cluster")[profile_features]
    .mean()
    .round(3)
)

cluster_sizes = (
    listener_profiles["cluster"]
    .value_counts()
    .sort_index()
)

display(cluster_summary)
display(cluster_sizes)


,clusters,silhouette_score,inertia
0,2,0.225797,6723.041913
1,3,0.196911,5767.622721
2,4,0.205689,5214.107035
3,5,0.174075,4763.565300
4,6,0.170867,4425.025397
5,7,0.164756,4148.564597
6,8,0.170082,3876.343270
7,9,0.178150,3709.467034
8,10,0.150522,3569.704863


,energy,danceability,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
cluster,,,,,,,,,
0,0.687,0.609,-7.854,0.127,0.189,0.194,0.218,0.527,118.962
1,0.585,0.514,-9.618,0.059,0.326,0.262,0.188,0.430,120.869
2,0.811,0.439,-6.314,0.081,0.082,0.191,0.221,0.396,127.166
3,0.704,0.528,-7.359,0.063,0.185,0.149,0.194,0.498,124.911


cluster
0     58
1    289
2    195
3    447
Name: count, dtype: int64

In [36]:
print('hello')

hello
